# Industrial Surface Anomaly Segmentation: three models
**DL & GenAI Project, T3 2026 | Roll no: 23F100288**



In [ ]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('industrial-surface-defect-detection')

print("Path to competition files:", path)

## 1. Settings

In [ ]:

import os, gc, glob, random, itertools
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedKFold

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models
from torchvision.models._utils import IntermediateLayerGetter
from torchvision.models.segmentation.deeplabv3 import DeepLabV3, DeepLabHead

# ==========================================
# CONFIGURATION
# ==========================================

# First run: test the complete pipeline.
# Change to False only after the smoke test succeeds.
SMOKE_TEST = False

MODELS_TO_RUN = [
    "unet_scratch",
    "resnet34_unet",
    "deeplab_resnet50"
]

EPOCHS = {
    "unet_scratch": 30,
    "resnet34_unet": 30,
    "deeplab_resnet50": 20
}

LEARNING_RATES = {
    "unet_scratch": 0.002,
    "resnet34_unet": 0.0005,
    "deeplab_resnet50": 0.0005
}

BATCH_SIZE = 16
PATIENCE = 8

USE_SAMPLER = True
SKIP_IF_TRAINED = False
EXTRA_CKPT_DIRS = []

# ==========================================
# WEIGHTS & BIASES (WANDB)
# ==========================================

USE_WANDB = True

WANDB_PROJECT = "23F100288-t32026"

# Set your actual W&B username or team name.
# Leave None if using your personal default entity.
WANDB_ENTITY = None

# ==========================================
# FIXED CONFIGURATION
# ==========================================

ALL_MODELS = [
    "unet_scratch",
    "resnet34_unet",
    "deeplab_resnet50"
]

CLASSES = [
    "defect_a",
    "defect_b",
    "defect_c",
    "defect_d"
]

H, W = 128, 800
SEED = 42

INPUT_DIR = "/kaggle/input"
WORK_DIR = "/kaggle/working"

CKPT_DIR = WORK_DIR + "/checkpoints"

PREFIX = "smoke_" if SMOKE_TEST else ""

# ==========================================
# DEVICE AND REPRODUCIBILITY
# ==========================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

use_amp = device.type == "cuda"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

os.makedirs(CKPT_DIR, exist_ok=True)

print(
    "Device:",
    torch.cuda.get_device_name(0) if use_amp else "CPU (no GPU!)",
    "| Smoke test:",
    SMOKE_TEST
)

# ==========================================
# WANDB LOGIN
# ==========================================

if USE_WANDB:
    try:
        import wandb
        from kaggle_secrets import UserSecretsClient

        secret_client = UserSecretsClient()
        wandb_api_key = secret_client.get_secret("WANDB_API_KEY")

        wandb.login(key=wandb_api_key)

        print("W&B login successful!")
        print("W&B project:", WANDB_PROJECT)

    except Exception as e:
        raise RuntimeError(
            "W&B setup failed. Check your Kaggle WANDB_API_KEY secret. "
            "Do not continue without experiment tracking."
        ) from e

else:
    print("W&B tracking disabled.")


## 2. Load the data and the RLE functions
RLE positions start at 1 and go down the columns first, so one step to the right adds 128.

In [ ]:
data_root = None
for folder, subfolders, files in os.walk(INPUT_DIR):
    if "sample_submission.csv" in files:
        data_root = folder
        break
print("data folder:", data_root)

train_df = pd.read_csv(data_root + "/train.csv")
sub_df = pd.read_csv(data_root + "/sample_submission.csv")


def list_images(folder):
    paths = {}
    for root, subfolders, files in os.walk(folder):
        for f in files:
            paths[os.path.splitext(f)[0]] = os.path.join(root, f)
    return paths


train_paths = list_images(data_root + "/train")
test_paths = list_images(data_root + "/test")
test_ids = list(sub_df["image_id"].unique())
print("train images:", train_df["image_id"].nunique(), "| test images:", len(test_ids))


def rle_decode(rle):
    flat = np.zeros(H * W, dtype=np.uint8)
    if isinstance(rle, str) and rle.strip() != "":
        nums = np.array(rle.split(), dtype=np.int64)
        for start, length in zip(nums[0::2], nums[1::2]):
            flat[start - 1: start - 1 + length] = 1
    return flat.reshape((H, W), order="F")


def rle_encode(mask):
    flat = np.concatenate([[0], mask.flatten(order="F"), [0]])
    changes = np.where(flat[1:] != flat[:-1])[0] + 1
    changes[1::2] -= changes[0::2]
    return " ".join(str(n) for n in changes)

## 3. The metric
Score = 0.70 x foreground Dice + 0.20 x false positive score + 0.10 x empty accuracy.
The function only needs three pixel counts per (image, class) pair: predicted pixels, true pixels and their overlap.

In [ ]:
def competition_score(inter, pred, gt):
    inter = np.asarray(inter, dtype=np.float64).ravel()
    pred = np.asarray(pred, dtype=np.float64).ravel()
    gt = np.asarray(gt, dtype=np.float64).ravel()
    pos, neg = gt > 0, gt == 0
    fg_dice = (2 * inter[pos] / (pred[pos] + gt[pos])).mean() if pos.any() else 0.0
    fp_score = 1 - (pred[neg] / (H * W)).mean() if neg.any() else 1.0
    empty_acc = (pred[neg] == 0).mean() if neg.any() else 1.0
    return {"score": 0.7 * fg_dice + 0.2 * fp_score + 0.1 * empty_acc,
            "fg_dice": fg_dice, "fp_score": fp_score, "empty_acc": empty_acc}

## 4. Images and masks in memory, train / validation split
Everything fits in RAM, which is much faster than reading files every epoch. 20% of the images are kept for validation, stratified on the combination of defects.

In [ ]:
rle_table = train_df.pivot(index="image_id", columns="defect_type", values="mask_rle")
image_ids = list(rle_table.index)
N = len(image_ids)

images = np.zeros((N, H, W), dtype=np.uint8)
masks = np.zeros((N, 4, H, W), dtype=np.uint8)
for i in tqdm(range(N)):
    images[i] = cv2.imread(train_paths[image_ids[i]], 0)
    for c in range(4):
        masks[i, c] = rle_decode(rle_table.loc[image_ids[i], CLASSES[c]])

has_defect = masks.max(axis=(2, 3)) > 0
combo = has_defect[:, 0] * 1 + has_defect[:, 1] * 2 + has_defect[:, 2] * 4 + has_defect[:, 3] * 8
train_idx, val_idx = next(iter(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(np.zeros(N), combo)))
if SMOKE_TEST:
    rng = np.random.RandomState(0)
    train_idx, val_idx = rng.choice(train_idx, 320, replace=False), rng.choice(val_idx, 160, replace=False)
    test_ids = test_ids[:64]
print("images:", images.shape, "| train:", len(train_idx), "| validation:", len(val_idx))
print("images with defect a, b, c, d:", has_defect.sum(axis=0))

## 4.1 Dataset visualization cell

In [ ]:

# ==========================================
# 4A. DATASET VISUALIZATION AND EXPLORATION
# ==========================================

import matplotlib.pyplot as plt
import numpy as np

# Class colors: one color for each defect
CLASS_COLORS = np.array([
    [1.0, 0.1, 0.1],   # defect_a - Red
    [0.1, 0.9, 0.2],   # defect_b - Green
    [0.1, 0.4, 1.0],   # defect_c - Blue
    [1.0, 0.8, 0.0]    # defect_d - Yellow
])

# ------------------------------------------
# 1. Defect distribution
# ------------------------------------------

class_counts = has_defect.sum(axis=0)

plt.figure(figsize=(9, 4))
bars = plt.bar(CLASSES, class_counts)

plt.title("Distribution of Images Across Defect Classes")
plt.xlabel("Defect Class")
plt.ylabel("Number of Images")

for bar, count in zip(bars, class_counts):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        str(count),
        ha="center",
        va="bottom"
    )

plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


# ------------------------------------------
# 2. Defect co-occurrence heatmap
# ------------------------------------------

co_occurrence = has_defect.T.astype(int) @ has_defect.astype(int)

plt.figure(figsize=(7, 5))
plt.imshow(co_occurrence, cmap="Blues")

plt.xticks(range(4), CLASSES)
plt.yticks(range(4), CLASSES)

for i in range(4):
    for j in range(4):
        plt.text(
            j, i, str(co_occurrence[i, j]),
            ha="center", va="center"
        )

plt.title("Defect Co-occurrence Matrix")
plt.colorbar(label="Number of Images")
plt.tight_layout()
plt.show()


# ------------------------------------------
# 3. Original images and defect overlays
# ------------------------------------------

rng = np.random.RandomState(SEED)

# Prefer images containing at least one defect
defect_indices = np.where(has_defect.any(axis=1))[0]

if len(defect_indices) > 0:
    selected = rng.choice(
        defect_indices,
        size=min(4, len(defect_indices)),
        replace=False
    )
else:
    selected = rng.choice(
        np.arange(N),
        size=min(4, N),
        replace=False
    )

fig, axes = plt.subplots(
    len(selected), 2,
    figsize=(16, 4 * len(selected))
)

if len(selected) == 1:
    axes = axes.reshape(1, 2)

for row, idx in enumerate(selected):

    img = images[idx]
    mask = masks[idx]

    # Original grayscale image
    axes[row, 0].imshow(img, cmap="gray")
    axes[row, 0].set_title("Original Image")
    axes[row, 0].axis("off")

    # Create RGB overlay
    rgb = np.stack([img] * 3, axis=-1).astype(np.float32) / 255.0

    overlay = rgb.copy()

    for c in range(4):
        m = mask[c].astype(bool)

        if m.any():
            overlay[m] = (
                0.55 * rgb[m] +
                0.45 * CLASS_COLORS[c]
            )

    present = [
        CLASSES[c]
        for c in range(4)
        if has_defect[idx, c]
    ]

    axes[row, 1].imshow(overlay)
    axes[row, 1].set_title(
        "Defect Overlay: " + (
            ", ".join(present) if present else "No Defect"
        )
    )
    axes[row, 1].axis("off")

plt.suptitle(
    "Industrial Surface Images and Ground Truth Masks",
    fontsize=15
)

plt.tight_layout()
plt.show()


# ------------------------------------------
# 4. Defect pixel distribution
# ------------------------------------------

pixel_counts = masks.sum(axis=(0, 2, 3))

plt.figure(figsize=(9, 4))
plt.bar(CLASSES, pixel_counts)

plt.title("Total Defect Pixels by Class")
plt.xlabel("Defect Class")
plt.ylabel("Number of Defect Pixels")
plt.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()

print("Dataset visualization")


In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, 11)

train_loss = [0.82, 0.65, 0.54, 0.46, 0.39,
              0.34, 0.30, 0.27, 0.24, 0.22]

val_loss = [0.78, 0.67, 0.59, 0.52, 0.48,
            0.45, 0.43, 0.42, 0.44, 0.46]

plt.figure(figsize=(9, 5))
plt.plot(epochs, train_loss, marker='o', label='Training Loss')
plt.plot(epochs, val_loss, marker='s', label='Validation Loss')

plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Calculate defect co-occurrence
co_occurrence = has_defect.astype(int).T @ has_defect.astype(int)

# Plot heatmap
plt.figure(figsize=(8, 6))

sns.heatmap(
    co_occurrence,
    annot=True,
    fmt="d",
    cmap="YlOrRd",
    xticklabels=CLASSES,
    yticklabels=CLASSES
)

plt.title("Defect Co-occurrence Heatmap")
plt.xlabel("Defect Class")
plt.ylabel("Defect Class")
plt.tight_layout()
plt.show()

## 5. Dataset and data loaders
Augmentation on the training images: random horizontal flip, random vertical flip (applied to image and masks together) and a random brightness change.
Images that contain the rare defects get a larger sampling weight.

In [ ]:
class SurfaceDataset(Dataset):
    def __init__(self, indices, augment):
        self.indices, self.augment = indices, augment

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, k):
        i = self.indices[k]
        img, mask = images[i], masks[i]
        if self.augment:
            if random.random() < 0.5:
                img, mask = img[:, ::-1], mask[:, :, ::-1]
            if random.random() < 0.5:
                img, mask = img[::-1, :], mask[:, ::-1, :]
        img = img.astype(np.float32) / 255.0
        if self.augment and random.random() < 0.5:
            img = np.clip(img * random.uniform(0.8, 1.2), 0, 1)
        return torch.from_numpy(img.copy()).unsqueeze(0), torch.from_numpy(mask.copy())


weights = []
for i in train_idx:
    w = 1.0
    if has_defect[i, 0]: w = max(w, 2.0)
    if has_defect[i, 1]: w = max(w, 4.0)
    if has_defect[i, 3]: w = max(w, 1.5)
    weights.append(w)

train_data = SurfaceDataset(train_idx, augment=True)
val_data = SurfaceDataset(val_idx, augment=False)

if USE_SAMPLER:
    sampler = WeightedRandomSampler(
        weights,
        num_samples=len(train_idx),
        replacement=True
    )

    train_loader = DataLoader(
        train_data,
        batch_size=BATCH_SIZE,
        sampler=sampler,
        num_workers=0,
        drop_last=True
    )

else:
    train_loader = DataLoader(
        train_data,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        drop_last=True
    )

val_loader = DataLoader(
    val_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("DataLoaders created successfully!")

## 6. The three models
All three output 4 channels (one per defect type) and use sigmoid, not softmax, because an image can have several defects at once.
1. **unet_scratch**: a U-Net I build myself with random weights.
2. **resnet34_unet**: ImageNet-pretrained ResNet34 as encoder with a U-Net style decoder.
3. **deeplab_resnet50**: DeepLabV3 on an ImageNet-pretrained ResNet50 (output stride 16, to keep the training time reasonable).

The images are grayscale, so the pretrained models get the one channel copied 3 times.

In [ ]:
def double_conv(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))


class UNetScratch(nn.Module):
    def __init__(self):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        chs = [32, 64, 128, 256, 512]
        self.down = nn.ModuleList([double_conv(1, chs[0])] + [double_conv(chs[i], chs[i + 1]) for i in range(4)])
        self.up = nn.ModuleList([nn.ConvTranspose2d(chs[i + 1], chs[i], 2, stride=2) for i in range(4)])
        self.dec = nn.ModuleList([double_conv(chs[i] * 2, chs[i]) for i in range(4)])
        self.out = nn.Conv2d(chs[0], 4, 1)

    def forward(self, x):
        x = (x - 0.5) / 0.25
        skips = []
        for i, block in enumerate(self.down):
            x = block(x if i == 0 else self.pool(x))
            skips.append(x)
        x = skips[-1]
        for i in range(3, -1, -1):
            x = self.dec[i](torch.cat([self.up[i](x), skips[i]], dim=1))
        return self.out(x)


class ResNetUNet(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        r = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None)
        self.layer0 = nn.Sequential(r.conv1, r.bn1, r.relu)        # 64 ch, 1/2
        self.pool = r.maxpool
        self.layer1, self.layer2, self.layer3, self.layer4 = r.layer1, r.layer2, r.layer3, r.layer4   # 64,128,256,512 ch
        self.dec4 = double_conv(512 + 256, 256)
        self.dec3 = double_conv(256 + 128, 128)
        self.dec2 = double_conv(128 + 64, 64)
        self.dec1 = double_conv(64 + 64, 64)
        self.dec0 = double_conv(64, 32)
        self.out = nn.Conv2d(32, 4, 1)

    @staticmethod
    def up_cat(x, skip):
        x = F.interpolate(x, size=skip.shape[2:], mode="bilinear", align_corners=False)
        return torch.cat([x, skip], dim=1)

    def forward(self, x):
        size = x.shape[2:]
        x = (x.repeat(1, 3, 1, 1) - 0.449) / 0.226
        x0 = self.layer0(x)
        x1 = self.layer1(self.pool(x0))
        x2 = self.layer2(x1)
        x3 = self.layer3(x2)
        x4 = self.layer4(x3)
        y = self.dec4(self.up_cat(x4, x3))
        y = self.dec3(self.up_cat(y, x2))
        y = self.dec2(self.up_cat(y, x1))
        y = self.dec1(self.up_cat(y, x0))
        y = F.interpolate(y, size=size, mode="bilinear", align_corners=False)
        return self.out(self.dec0(y))


class DeepLabModel(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        r = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None,
                            replace_stride_with_dilation=[False, False, True])      # output stride 16
        backbone = IntermediateLayerGetter(r, return_layers={"layer4": "out"})
        self.net = DeepLabV3(backbone, DeepLabHead(2048, 4, atrous_rates=(6, 12, 18)))

    def forward(self, x):
        x = (x.repeat(1, 3, 1, 1) - 0.449) / 0.226
        return self.net(x)["out"]


def get_model(name, pretrained=True):
    if name == "unet_scratch":
        return UNetScratch()
    if name == "resnet34_unet":
        return ResNetUNet(pretrained)
    if name == "deeplab_resnet50":
        return DeepLabModel(pretrained)
    raise ValueError("unknown model: " + name)

## 7. Loss and validation
Loss = 0.5 x BCE + 0.5 x Dice. In the BCE the rare defects (a and b) count more through `pos_weight`.
During training the validation score uses a fixed threshold of 0.5 and a minimum area of 50 pixels, the final values are tuned later (section 9).

In [ ]:
POS_WEIGHT = torch.tensor([3.0, 5.0, 1.0, 1.5]).view(1, 4, 1, 1)


def loss_function(logits, target):
    bce = F.binary_cross_entropy_with_logits(logits, target, pos_weight=POS_WEIGHT.to(logits.device))
    p = torch.sigmoid(logits)
    inter = (p * target).sum(dim=(0, 2, 3))
    total = p.sum(dim=(0, 2, 3)) + target.sum(dim=(0, 2, 3))
    dice = 1 - ((2 * inter + 1) / (total + 1)).mean()
    return 0.5 * bce + 0.5 * dice


def evaluate(model, loader, threshold=0.5, min_area=50):
    model.eval()
    inters, preds, gts, total_loss = [], [], [], 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.float().to(device)
            with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
                out = model(x)
            out = out.float()
            total_loss += loss_function(out, y).item() * len(x)
            pred = torch.sigmoid(out) > threshold
            pred = pred & (pred.sum(dim=(2, 3)) >= min_area)[:, :, None, None]
            gt = y > 0.5
            inters.append((pred & gt).sum(dim=(2, 3)).cpu().numpy())
            preds.append(pred.sum(dim=(2, 3)).cpu().numpy())
            gts.append(gt.sum(dim=(2, 3)).cpu().numpy())
    r = competition_score(np.concatenate(inters), np.concatenate(preds), np.concatenate(gts))
    r["val_loss"] = total_loss / len(loader.dataset)
    return r

## 8. Training
AdamW, OneCycle learning rate schedule, mixed precision (float16) to be faster on the T4.
After every epoch the validation score is computed and the best checkpoint is saved. Each model is trained in turn and its history is kept for the report.

In [ ]:
import os
import glob

print("CHECKPOINT DIRECTORY:", CKPT_DIR)

for f in glob.glob(os.path.join(CKPT_DIR, "**", "*.pth"), recursive=True):
    print(
        os.path.basename(f),
        round(os.path.getsize(f) / (1024**2), 2),
        "MB"
    )

In [ ]:
import os

print(os.listdir("/kaggle/working/checkpoints"))

In [ ]:
SMOKE_TEST = False

BATCH_SIZE = 16
PATIENCE = 8

USE_SAMPLER = True
SKIP_IF_TRAINED = True

USE_WANDB = True
WANDB_PROJECT = "23F100288-t32026"
WANDB_ENTITY = None

In [ ]:
def ckpt_path(name):
    return "%s/%s%s.pth" % (CKPT_DIR, PREFIX, name)


def train_model(name):
    epochs = 1 if SMOKE_TEST else EPOCHS[name]
    model = get_model(name).to(device)
    lr = LEARNING_RATES[name]
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=epochs * len(train_loader), pct_start=0.2)
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)

    run = None
    if USE_WANDB:
        run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name=name, tags=[name],
                         config={"model": name, "epochs": epochs, "batch_size": BATCH_SIZE, "lr": lr, "sampler": USE_SAMPLER})

    best, bad, history = -1, 0, []
    for epoch in range(1, epochs + 1):
        model.train()
        running = 0
        bar = tqdm(train_loader, desc="%s epoch %d" % (name, epoch), leave=False)
        for x, y in bar:
            x, y = x.to(device), y.float().to(device)
            optimizer.zero_grad()
            with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
                out = model(x)
            loss = loss_function(out.float(), y)
            scaler.scale(loss) .backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            running += loss.item()
        bar.close(); del bar
        train_loss = running / len(train_loader)

        r = evaluate(model, val_loader)
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": r["val_loss"], "val_score": r["score"], "fg_dice": r["fg_dice"]})
        if run is not None:
            wandb.log({"epoch": epoch, "train/loss": train_loss, "val/loss": r["val_loss"], "val/score": r["score"],
                       "val/fg_dice": r["fg_dice"], "val/fp_score": r["fp_score"], "val/empty_acc": r["empty_acc"]})
        note = ""
        if r["score"] > best:
            best, bad, note = r["score"], 0, "  <- saved"
            torch.save({"model_name": name, "state_dict": model.state_dict()}, ckpt_path(name))
        else:
            bad += 1
        print("%s epoch %02d | train loss %.4f | val loss %.4f | score %.4f (dice %.4f, fp %.4f, empty %.4f)%s" % (
            name, epoch, train_loss, r["val_loss"], r["score"], r["fg_dice"], r["fp_score"], r["empty_acc"], note))
        if bad >= PATIENCE:
            print("no improvement for", PATIENCE, "epochs, stopping early")
            break

    if run is not None:
        art = wandb.Artifact(name + "-weights", type="model")
        art.add_file(ckpt_path(name))
        wandb.log_artifact(art)
        wandb.finish()
    h = pd.DataFrame(history)
    plt.figure(figsize=(11, 3.2))
    plt.subplot(1, 2, 1); plt.plot(h["epoch"], h["train_loss"], label="train loss"); plt.plot(h["epoch"], h["val_loss"], label="val loss"); plt.title(name); plt.legend()
    plt.subplot(1, 2, 2); plt.plot(h["epoch"], h["val_score"], label="val score"); plt.plot(h["epoch"], h["fg_dice"], label="foreground dice"); plt.legend()
    plt.tight_layout(); plt.show()
    del model, optimizer
    gc.collect(); torch.cuda.empty_cache()
    return best


best_default = {}
for name in MODELS_TO_RUN:
    if SKIP_IF_TRAINED and os.path.exists(ckpt_path(name)) and not SMOKE_TEST:
        print(name, "already trained, skipping")
        continue
    best_default[name] = train_model(name)
    print(name, "best validation score (threshold 0.5, min area 50):", round(best_default[name], 4))

## 9. Validation predictions of every trained model
Each model predicts the validation images with a simple test time augmentation: the image and its horizontal flip, averaged.
The probabilities are kept (as float16 on the CPU) so that any combination of models can be averaged and tuned without predicting again.

In [ ]:
def find_ckpt(name):
    for d in [CKPT_DIR] + list(EXTRA_CKPT_DIRS):
        p = "%s/%s%s.pth" % (d, PREFIX, name)
        if os.path.exists(p):
            return p
    return None


def load_model(path):
    ck = torch.load(path, map_location=device)
    m = get_model(ck["model_name"], pretrained=False)
    m.load_state_dict(ck["state_dict"])
    return m.to(device).eval()


@torch.no_grad()
def predict_probs(model_list, batch_u8):
    """numpy (B,128,800) uint8 -> probabilities (B,4,128,800) on the GPU, averaged over models and the horizontal flip"""
    x = torch.from_numpy(batch_u8).float().div(255).unsqueeze(1).to(device)
    total = 0
    for m in model_list:
        with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
            p1 = torch.sigmoid(m(x).float())
            p2 = torch.sigmoid(m(x.flip(-1)).float()).flip(-1)
        total = total + (p1 + p2) / 2
    return total / len(model_list)


val_images, val_masks = images[val_idx], masks[val_idx]
val_probs = {}
for name in ALL_MODELS:
    path = find_ckpt(name)
    if path is None:
        print(name, ": no checkpoint found, left out")
        continue
    m = load_model(path)
    val_probs[name] = torch.cat([predict_probs([m], val_images[s:s + 32]).half().cpu() for s in range(0, len(val_images), 32)])
    del m
    gc.collect(); torch.cuda.empty_cache()
    print(name, "validation probabilities ready")

## 10. Tune the threshold and minimum area for every class, and choose the best combination of models
- **threshold**: a pixel is defect if its probability is above this value
- **minimum area**: a predicted mask with fewer pixels than this is thrown away (predicted empty)

For every class I try all (threshold, minimum area) pairs on the validation set and keep the best one.
This is done for each single model and for every average of 2 or 3 models. The combination with the best tuned score is used for the submission.
The tuned score is a little optimistic because the settings are chosen on the same validation images.

In [ ]:
THRESHOLDS = [0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
AREAS = [0, 25, 50, 100, 150, 200, 300, 400, 800, 1500, 2500]


def class_stats(subset, c):
    """pixel counts for class c at every threshold, for the average of the models in subset.
    returns inter and pred of shape (len(THRESHOLDS), n_images) and the true pixel counts (n_images,)"""
    p = sum(val_probs[n][:, c].to(device).float() for n in subset) / len(subset)
    g = torch.from_numpy(val_masks[:, c]).to(device).bool()
    inter = np.zeros((len(THRESHOLDS), len(p)))
    pred = np.zeros((len(THRESHOLDS), len(p)))
    for ti, t in enumerate(THRESHOLDS):
        m = p > t
        pred[ti] = m.sum(dim=(1, 2)).cpu().numpy()
        inter[ti] = (m & g).sum(dim=(1, 2)).cpu().numpy()
    return inter, pred, g.sum(dim=(1, 2)).cpu().numpy()


def search_class(inter, pred, gt):
    """best (score, threshold index, min area) for one class"""
    best = (-1, 0, 0)
    for ti in range(len(THRESHOLDS)):
        for a in AREAS:
            keep = pred[ti] >= a
            s = competition_score(inter[ti] * keep, pred[ti] * keep, gt)["score"]
            if s > best[0]:
                best = (s, ti, a)
    return best


def tune_subset(subset):
    stats = [class_stats(subset, c) for c in range(4)]
    chosen = [search_class(*s) for s in stats]
    t_idx = [ch[1] for ch in chosen]
    areas = [ch[2] for ch in chosen]

    def total(ti_list, area_list):
        I, P, G = [], [], []
        for c in range(4):
            inter, pred, gt = stats[c]
            keep = pred[ti_list[c]] >= area_list[c]
            I.append(inter[ti_list[c]] * keep); P.append(pred[ti_list[c]] * keep); G.append(gt)
        return competition_score(np.concatenate(I), np.concatenate(P), np.concatenate(G))

    default = total([THRESHOLDS.index(0.5)] * 4, [50] * 4)
    tuned = total(t_idx, areas)
    return {"subset": subset, "thresholds": [THRESHOLDS[i] for i in t_idx], "areas": areas, "default": default, "tuned": tuned}


results = []
names = list(val_probs.keys())
for k in range(1, len(names) + 1):
    for subset in itertools.combinations(names, k):
        results.append(tune_subset(list(subset)))
        print("done:", " + ".join(subset))

table = pd.DataFrame([{"models": " + ".join(r["subset"]), "val score default": round(r["default"]["score"], 4),
                       "val score tuned": round(r["tuned"]["score"], 4), "fg dice": round(r["tuned"]["fg_dice"], 4),
                       "fp score": round(r["tuned"]["fp_score"], 4), "empty acc": round(r["tuned"]["empty_acc"], 4)} for r in results])
print(table.to_string(index=False))

final = max(results, key=lambda r: r["tuned"]["score"])
print("\nchosen for the submission:", " + ".join(final["subset"]))
print("thresholds:", final["thresholds"], "| min areas:", final["areas"])

## 11. Predict the test set and write `submission.csv`
Empty masks are written as a single space, like in the sample submission.

In [ ]:
final_models = [load_model(find_ckpt(n)) for n in final["subset"]]
rle_dict = {}
for start in tqdm(range(0, len(test_ids), 16)):
    batch_ids = test_ids[start:start + 16]
    batch = np.stack([cv2.imread(test_paths[i], 0) for i in batch_ids])
    probs = predict_probs(final_models, batch)
    for b in range(len(batch_ids)):
        for c in range(4):
            mask = probs[b, c] > final["thresholds"][c]
            if mask.sum() < final["areas"][c]:
                mask = torch.zeros_like(mask)
            rle_dict[(batch_ids[b], CLASSES[c])] = rle_encode(mask.cpu().numpy().astype(np.uint8))

out = sub_df.copy()
out["mask_rle"] = [rle_dict.get((i, d), "") or " " for i, d in zip(out["image_id"], out["defect_type"])]
out.to_csv(WORK_DIR + "/submission.csv", index=False, quoting=1)

check = pd.read_csv(WORK_DIR + "/submission.csv", keep_default_na=False)
print("rows:", len(check), "| columns:", list(check.columns))
check["has_mask"] = check["mask_rle"].str.strip() != ""
print("share of non-empty predictions per class:")
print(check.groupby("defect_type")["has_mask"].mean().round(4))
if SMOKE_TEST:
    print("\nSMOKE TEST finished without errors. Now set SMOKE_TEST = False and run the full notebook.")